# Algoritmos de búsqueda en grafos

En este notebook trabajaremos sobre **un mismo problema** para comparar:

- Depth-First Search (DFS)
- Breadth-First Search (BFS)
- Uniform Cost Search (UCS)
- Greedy Best-First Search (GBF)
- A*

La idea central es observar que la estructura general de búsqueda cambia poco.  
Lo que cambia principalmente es **cómo se organiza la frontier**.

## 1. Problema de búsqueda

- Estado inicial: `A`
- Estado meta: `F`
- Los números en las aristas representan el costo de moverse entre estados.

![Grafo DFS](https://drive.google.com/uc?export=view&id=1Bu3wvTzHffMQVHmGfvekSdmtaatt12qJ)



## 2. Representación del problema

Como el grafo tiene costos, cada vecino se representa mediante una tupla:

```python
(estado_vecino, costo_de_la_arista)
```

El orden de los vecinos también importa para DFS y BFS.

In [8]:
graph = {
    "A": [("B", 2), ("C", 1)],
    "B": [("D", 2), ("E", 1)],
    "C": [("E", 3)],
    "D": [("H", 1)],
    "E": [("H", 2)],
    "H": []
}

start = "A"
goal = "H"

## 3. Estructuras comunes

Cada nodo de búsqueda guarda:

- `state`: estado actual.
- `parent`: nodo desde el cual se llegó.
- `cost`: costo acumulado \(g(n)\).

In [9]:
class Node:
    def __init__(self, state, parent=None, cost=0):
        self.state = state
        self.parent = parent
        self.cost = cost

    def __repr__(self):
        return f"{self.state}(g={self.cost})"

In [11]:
def reconstruct_path(node):
    path = []

    while node is not None:
        path.append(node.state)
        node = node.parent

    return list(reversed(path))

## 4. DFS

DFS utiliza una **pila LIFO**.

Los costos aparecen en el grafo, pero DFS no los utiliza para decidir qué nodo expandir.

In [2]:
class StackFrontier:
    def __init__(self):
        self.frontier = []

    def add(self, node):
        self.frontier.append(node)

    def contains_state(self, state):
        return any(node.state == state for node in self.frontier)

    def empty(self):
        return len(self.frontier) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        return self.frontier.pop()

    def states(self):
        return [node.state for node in self.frontier]

In [12]:
def depth_first_search(graph, start, goal, verbose=True):
    frontier = StackFrontier()
    frontier.add(Node(start))

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node = frontier.remove()
        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo: {node.state}")
            print(f"Frontier antes de agregar vecinos: {frontier.states()}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            if (
                child_state not in explored
                and not frontier.contains_state(child_state)
            ):
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )
                frontier.add(child)

        if verbose:
            print(f"Frontier después de expandir: {frontier.states()}")
            print("-" * 45)

    return None

In [13]:
dfs_result = depth_first_search(graph, start, goal)

print("Orden de expansión:", " → ".join(dfs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(dfs_result["path"]))
print("Costo del camino:", dfs_result["cost"])

Expandiendo: A
Frontier antes de agregar vecinos: []
Frontier después de expandir: ['B', 'C']
---------------------------------------------
Expandiendo: C
Frontier antes de agregar vecinos: ['B']
Frontier después de expandir: ['B', 'E']
---------------------------------------------
Expandiendo: E
Frontier antes de agregar vecinos: ['B']
Frontier después de expandir: ['B', 'H']
---------------------------------------------
Expandiendo: H
Frontier antes de agregar vecinos: ['B']
Orden de expansión: A → C → E → H
Camino encontrado: A → C → E → H
Costo del camino: 6


## 5. BFS — actividad

BFS utiliza una **cola FIFO**.

### Tareas

1. Complete el método `remove` de `QueueFrontier`.
2. Complete `breadth_first_search`.
3. Verifique el orden de expansión y el camino encontrado.
4. Compare el resultado con su solución manual.

In [ ]:
class QueueFrontier(StackFrontier):
    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        # Única diferencia con StackFrontier: se extrae por el frente en lugar
        # del final, de modo que sale el nodo que lleva más tiempo esperando.
        # El resto del comportamiento (add, contains_state, empty) se hereda.
        return self.frontier.pop(0)

In [ ]:
def breadth_first_search(graph, start, goal, verbose=True):
    """BFS: la misma estructura de DFS, cambiando la pila por una cola FIFO.

    Al expandir siempre el nodo más antiguo, el grafo se recorre por niveles.
    Eso garantiza que el primer camino encontrado sea el de menor número de
    aristas, que no es necesariamente el de menor costo.
    """
    frontier = QueueFrontier()
    frontier.add(Node(start))

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node = frontier.remove()
        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo: {node.state}")
            print(f"Frontier antes de agregar vecinos: {frontier.states()}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            # Se descartan los estados ya expandidos y los que ya esperan turno
            # en la frontier: evita repetir trabajo y entrar en ciclos.
            if (
                child_state not in explored
                and not frontier.contains_state(child_state)
            ):
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )
                frontier.add(child)

        if verbose:
            print(f"Frontier después de expandir: {frontier.states()}")
            print("-" * 45)

    return None

In [ ]:
# Ejecute esta celda cuando complete BFS.
bfs_result = breadth_first_search(graph, start, goal)

print("Orden de expansión:", " → ".join(bfs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(bfs_result["path"]))
print("Costo del camino:", bfs_result["cost"])

In [ ]:
# Pruebas básicas para BFS
assert bfs_result["path"] == ["A", "B", "D", "H"]
assert bfs_result["expansion_order"] == ["A", "B", "C", "D", "E", "H"]

print("✓ BFS pasó las pruebas.")

## 6. UCS

UCS utiliza una **cola de prioridad** y siempre expande el nodo con menor costo acumulado:

$$
g(n)
$$

Si aparece un camino más barato hacia un estado que ya estaba en la frontier, se conserva el de menor costo.

### Frontier con prioridad

En **Uniform Cost Search (UCS)**, **Greedy Best-First Search (GBF)** y **A\*** la *frontier* ya no es una pila ni una cola, sino una **cola de prioridad**.

La prioridad determina cuál será el siguiente nodo en expandirse:

- **UCS:** prioridad = \(g(n)\)
- **GBF:** prioridad = \(h(n)\)
- **A\*:** prioridad = \(g(n)+h(n)\)

Esta implementación utiliza el módulo `heapq` de Python, que mantiene automáticamente el elemento con **menor prioridad** en la primera posición.

#### Componentes de la clase

- **`heap`**: almacena la cola de prioridad.
- **`counter`**: genera un número consecutivo para cada inserción. Se utiliza para desempatar cuando dos nodos tienen la misma prioridad, respetando el orden en que fueron agregados.
- **`add(node, priority)`**: inserta un nodo en la frontier con su prioridad correspondiente.
- **`remove()`**: extrae el nodo con la menor prioridad.
- **`empty()`**: indica si la frontier está vacía.

Cada elemento del heap se almacena como una tupla:

```python
(priority, insertion_order, node)
```

Por ejemplo,

```python
(5, 3, Node("E"))
```

significa:

- prioridad = **5**
- fue el **cuarto nodo** insertado (`3` porque el contador empieza en 0)
- corresponde al nodo **E**.

De esta forma, si dos nodos tienen la misma prioridad, se expandirá primero el que fue insertado antes.

### ¿Qué hace `heapq.heappush`?

La siguiente instrucción agrega un nuevo nodo a la **cola de prioridad** (`heap`):

```python
heapq.heappush(
    self.heap,
    (priority, next(self.counter), node)
)
```

Observa que **no se almacena únicamente el nodo**, sino una **tupla** con tres elementos:

```python
(priority, insertion_order, node)
```

donde:

- **`priority`**: valor utilizado para ordenar la frontier.
  - UCS: `g(n)`
  - GBF: `h(n)`
  - A*: `g(n) + h(n)`

- **`next(self.counter)`**: número consecutivo que indica el orden de inserción.
  Se utiliza para desempatar cuando dos nodos tienen la misma prioridad.

- **`node`**: objeto que contiene el estado, el padre y el costo acumulado.

---

### Ejemplo

Supongamos que insertamos los siguientes nodos:

```python
(5, 0, Node("B"))
(3, 1, Node("C"))
(5, 2, Node("D"))
```

La prioridad es el **primer elemento** de la tupla, por lo que el primer nodo en salir será:

```text
Node("C")
```

porque tiene prioridad **3**.

Posteriormente saldrán:

```text
Node("B")
Node("D")
```

Ambos tienen prioridad **5**, pero `B` fue insertado antes (`0 < 2`).

---

### ¿Por qué usar el contador?

Si almacenáramos únicamente

```python
(priority, node)
```

y dos nodos tuvieran la misma prioridad, Python intentaría comparar directamente los objetos `Node`, lo que produciría un error.

El contador evita este problema y garantiza un criterio de desempate consistente.

In [14]:
import heapq
from itertools import count

In [15]:
class PriorityFrontier:
    def __init__(self):
        self.heap = []
        self.counter = count()

    def add(self, node, priority):
        # El contador permite desempatar respetando el orden de inserción.
        heapq.heappush(
            self.heap,
            (priority, next(self.counter), node)
        )

    def empty(self):
        return len(self.heap) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        priority, _, node = heapq.heappop(self.heap)
        return node, priority

In [16]:
def uniform_cost_search(graph, start, goal, verbose=True):
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=0)

    # Mejor costo conocido para cada estado.
    best_cost = {start: 0}

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        # Ignorar entradas antiguas de la cola de prioridad.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost)

    return None

In [17]:
ucs_result = uniform_cost_search(graph, start, goal)

print("Orden de expansión:", " → ".join(ucs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(ucs_result["path"]))
print("Costo óptimo:", ucs_result["cost"])

Expandiendo A: g=0
Expandiendo C: g=1
Expandiendo B: g=2
Expandiendo E: g=3
Expandiendo D: g=4
Expandiendo H: g=5
Orden de expansión: A → C → B → E → D → H
Camino encontrado: A → B → E → H
Costo óptimo: 5


## 7. Heurística para GBF y A*

La heurística \(h(n)\) estima el costo restante desde cada nodo hasta la meta.

![Heurística](heuristica_busqueda.png)

In [18]:
heuristic = {
    "A": 4,
    "B": 3,
    "C": 2,
    "D": 2,
    "E": 1,
    "H": 0
}

### Verificación de la heurística

Antes de usarla conviene revisar dos propiedades, porque de ellas dependen las
garantías de A*:

- **Admisible:** $h(n) \le h^*(n)$, donde $h^*(n)$ es el costo real mínimo de $n$ a la meta.
  Si se cumple, A* devuelve el camino óptimo.
- **Consistente:** $h(n) \le c(n,m) + h(m)$ para toda arista $n \to m$.
  Si se cumple, ningún estado necesita reexpandirse.

Para obtener $h^*$ se aplica Dijkstra sobre el **grafo invertido** partiendo de
la meta: así, de una sola pasada, se calcula la distancia real de cada nodo a la
meta y se puede comparar contra la heurística propuesta.

In [ ]:
def true_costs(graph, goal):
    """Costo real mínimo de cada nodo hasta la meta.

    Se invierte el grafo y se corre Dijkstra desde la meta: recorrer las aristas
    al revés convierte "distancia desde la meta" en "distancia hasta la meta",
    que es justamente el h*(n) contra el cual se compara la heurística.
    """
    reverse = {}

    for origin, edges in graph.items():
        reverse.setdefault(origin, [])
        for destination, weight in edges:
            reverse.setdefault(destination, []).append((origin, weight))

    dist = {state: float("inf") for state in reverse}
    dist[goal] = 0
    heap = [(0, goal)]

    while heap:
        d, state = heapq.heappop(heap)

        # Entrada obsoleta del heap: ya se registró una distancia menor.
        if d > dist[state]:
            continue

        for parent, weight in reverse[state]:
            if d + weight < dist[parent]:
                dist[parent] = d + weight
                heapq.heappush(heap, (dist[parent], parent))

    return dist


def check_heuristic(graph, heuristic, goal):
    """Reporta las violaciones de admisibilidad y consistencia.

    Devuelve True solo si la heurística cumple ambas propiedades.
    """
    h_star = true_costs(graph, goal)

    no_admisibles = [
        (state, heuristic[state], h_star[state])
        for state in sorted(h_star)
        if heuristic[state] > h_star[state]
    ]

    no_consistentes = [
        (origin, destination, heuristic[origin], weight + heuristic[destination])
        for origin, edges in graph.items()
        for destination, weight in edges
        if heuristic[origin] > weight + heuristic[destination]
    ]

    print("Diagnóstico de la heurística")
    print("-" * 45)

    if no_admisibles:
        for state, h, real in no_admisibles:
            print(f"  No admisible en {state}: h={h} > h*={real}")
    else:
        print("  Admisible: h(n) <= h*(n) en todos los nodos.")

    if no_consistentes:
        for origin, destination, h, cota in no_consistentes:
            print(f"  No consistente en {origin}->{destination}: h({origin})={h} > {cota}")
    else:
        print("  Consistente en todas las aristas.")

    if no_admisibles:
        print("\n  Con esta heurística A* no tiene garantía formal de optimalidad.")

    return not no_admisibles and not no_consistentes

In [ ]:
check_heuristic(graph, heuristic, goal)

## 8. Greedy Best-First Search — actividad

GBF utiliza únicamente:

$$
h(n)
$$

### Tareas

1. Reutilice `PriorityFrontier`.
2. Use la heurística como prioridad.
3. No use el costo acumulado para decidir qué nodo expandir.
4. Retorne el camino, orden de expansión y costo real del camino.

In [ ]:
def greedy_best_first_search(graph, heuristic, start, goal, verbose=True):
    """GBF: la prioridad es únicamente h(n), la estimación de lo que falta.

    A diferencia de UCS, nunca consulta el costo ya recorrido para decidir.
    El costo se sigue acumulando en cada nodo, pero solo para poder reportar
    al final el costo real del camino, no para ordenar la frontier.
    """
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=heuristic[start])

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, _ = frontier.remove()

        # Un mismo estado puede haber entrado por varias ramas. Como aquí no se
        # comparan costos, basta con quedarse con la primera vez que sale.
        if node.state in explored:
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: h={heuristic[node.state]}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            if child_state not in explored:
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )

                # La prioridad ignora child.cost: ahí está el sesgo de GBF.
                frontier.add(child, priority=heuristic[child_state])

    return None

In [ ]:
# Ejecute esta celda cuando complete GBF.
gbf_result = greedy_best_first_search(graph, heuristic, start, goal)

print("Orden de expansión:", " → ".join(gbf_result["expansion_order"]))
print("Camino encontrado:", " → ".join(gbf_result["path"]))
print("Costo del camino:", gbf_result["cost"])

In [ ]:
# Pruebas básicas para GBF
assert gbf_result["path"] == ["A", "C", "E", "H"]
assert gbf_result["expansion_order"] == ["A", "C", "E", "H"]
assert gbf_result["cost"] == 6

print("✓ GBF pasó las pruebas.")

## 9. A* — actividad

A* combina costo acumulado y heurística:

$$
f(n)=g(n)+h(n)
$$

### Tareas

1. Adapte la implementación de UCS.
2. Mantenga `best_cost` para permitir mejoras de ruta.
3. Use `new_cost + heuristic[child_state]` como prioridad.
4. Verifique que encuentre un camino óptimo.

In [ ]:
def a_star_search(graph, heuristic, start, goal, verbose=True):
    """A*: la prioridad es f(n) = g(n) + h(n), lo recorrido más lo estimado.

    Estructuralmente es UCS con la heurística sumada a la prioridad. Se conserva
    best_cost por dos razones: descartar entradas obsoletas de la cola y volver
    a encolar un estado cuando aparece una ruta más barata hacia él, lo que
    protege el resultado incluso si la heurística no es consistente.
    """
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=heuristic[start])

    # Mejor costo conocido para cada estado.
    best_cost = {start: 0}

    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        # heapq no permite actualizar prioridades, así que las versiones caras
        # de un estado siguen en el heap. Se reconocen porque su g ya no coincide
        # con el mejor costo registrado, y se descartan al salir.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}, f={priority}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            # Solo se encola si mejora la ruta conocida hacia ese estado.
            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost + heuristic[child_state])

    return None

In [ ]:
# Ejecute esta celda cuando complete A*.
astar_result = a_star_search(graph, heuristic, start, goal)

print("Orden de expansión:", " → ".join(astar_result["expansion_order"]))
print("Camino encontrado:", " → ".join(astar_result["path"]))
print("Costo óptimo:", astar_result["cost"])

In [ ]:
# Pruebas básicas para A*
assert astar_result["path"] in (
    ["A", "B", "D", "H"],
    ["A", "B", "E", "H"]
)
assert astar_result["cost"] == 5

print("✓ A* pasó las pruebas.")

## 10. Comparación final

Los cinco algoritmos comparten el mismo esqueleto: sacar un nodo de la frontier,
comprobar si es la meta, marcarlo como explorado y encolar sus vecinos. Lo único
que cambia es el criterio con el que la frontier decide quién sale primero.

En lugar de copiar los resultados a mano, se recogen ejecutando todos los
algoritmos sobre el mismo problema y tabulándolos.

In [ ]:
def compare(graph, start, goal, heuristic=None):
    """Ejecuta los algoritmos sobre el mismo problema y los tabula.

    UCS sirve de referencia de optimalidad: como siempre expande por g(n) menor,
    el costo que reporta es el mínimo alcanzable, y contra él se marca cuáles de
    los demás algoritmos lo igualan.

    Si no se pasa heurística se omiten GBF y A*, que dependen de ella.
    """
    runs = [
        ("DFS", lambda: depth_first_search(graph, start, goal, verbose=False)),
        ("BFS", lambda: breadth_first_search(graph, start, goal, verbose=False)),
        ("UCS", lambda: uniform_cost_search(graph, start, goal, verbose=False)),
    ]

    if heuristic:
        runs += [
            ("GBF", lambda: greedy_best_first_search(graph, heuristic, start, goal, verbose=False)),
            ("A*", lambda: a_star_search(graph, heuristic, start, goal, verbose=False)),
        ]

    results = [(name, run()) for name, run in runs]

    optimal = next(
        (result["cost"] for name, result in results if name == "UCS" and result),
        None
    )

    print(f"{'Algoritmo':<11}{'Expandidos':>11}{'Costo':>7}{'Óptimo':>8}   Camino")
    print("-" * 62)

    for name, result in results:
        # None significa que la meta no es alcanzable desde el inicio.
        if result is None:
            print(f"{name:<11}{'-':>11}{'-':>7}{'-':>8}   sin solución")
            continue

        optimo = "sí" if result["cost"] == optimal else "no"

        print(
            f"{name:<11}{len(result['expansion_order']):>11}"
            f"{result['cost']:>7}{optimo:>8}   {' → '.join(result['path'])}"
        )

    return results

In [ ]:
comparison = compare(graph, start, goal, heuristic)

### Resultados

| Algoritmo | Tipo de frontier | Prioridad | Camino | Costo | Orden de expansión |
|---|---|---|---|---:|---|
| DFS | Pila | LIFO | A → C → E → H | 6 | A → C → E → H |
| BFS | Cola | FIFO | A → B → D → H | 5 | A → B → C → D → E → H |
| UCS | Cola de prioridad | \(g(n)\) | A → B → E → H | 5 | A → C → B → E → D → H |
| GBF | Cola de prioridad | \(h(n)\) | A → C → E → H | 6 | A → C → E → H |
| A* | Cola de prioridad | \(g(n)+h(n)\) | A → B → E → H | 5 | A → C → B → E → H |

Dos observaciones sobre la tabla:

- DFS y GBF llegan al mismo camino por razones distintas. DFS toma `C` porque fue
  el último vecino apilado; GBF lo toma porque `h(C)=2 < h(B)=3`. Coinciden por el
  tamaño del grafo, no por parentesco entre los algoritmos.
- A* expande cinco nodos y UCS seis para llegar al mismo costo. La heurística no
  cambia el resultado, pero evita expandir `D`: ese es todo su aporte aquí.

### Preguntas de cierre

**1. ¿Qué algoritmos garantizan el camino de menor número de aristas?**

Solo BFS. Al vaciar la frontier por niveles, la primera vez que alcanza la meta lo
hace por un camino de longitud mínima en aristas. En la tabla encuentra `A → B → D → H`,
de tres aristas. UCS coincide únicamente en el caso particular en que todas las
aristas tienen el mismo costo, porque ahí el costo acumulado equivale a contar aristas.

**2. ¿Qué algoritmos garantizan el camino de menor costo?**

UCS siempre, porque expande en orden creciente de \(g(n)\): cuando extrae la meta, ya
descartó cualquier ruta más barata. A* lo garantiza solo si la heurística es admisible.

Aquí conviene ser preciso: `check_heuristic` muestra que la heurística del enunciado
**no** es admisible (`h(D)=2` frente a un costo real de 1) ni consistente
(`A→C`, `B→E` y `D→H` violan la desigualdad). A* llega al costo óptimo 5 en este grafo,
pero por el tamaño del ejemplo, no porque la teoría lo asegure. Mantener `best_cost` y
permitir reencolar un estado con \(g\) menor es lo que evita que la inconsistencia
degrade el resultado.

**3. ¿Por qué GBF puede encontrar un camino subóptimo?**

Porque su prioridad ignora por completo lo ya recorrido. Desde `A` compara `h(B)=3`
contra `h(C)=2` y baja por `C`, sin ver que llegar a `C` es barato pero salir de él
cuesta 3. Termina con costo 6 frente al óptimo de 5. La heurística mide lo que falta,
no lo que se lleva gastado, y sin \(g(n)\) no hay forma de corregir una decisión temprana.

**4. ¿Qué ocurre con A* si \(h(n)=0\) para todos los nodos?**

\(f(n) = g(n) + 0 = g(n)\), así que la frontier se ordena exactamente igual que en UCS
y A* se reduce a UCS. Es el caso extremo de heurística admisible: nunca sobreestima,
pero tampoco aporta información, de modo que se conserva la optimalidad y se pierde la
capacidad de podar. En la práctica se ve como un aumento de nodos expandidos.

**5. ¿Qué efecto tiene el orden de los vecinos en DFS y BFS?**

En DFS es determinante. Como la pila devuelve el último vecino apilado, invertir
`"A": [("B", 2), ("C", 1)]` a `[("C", 1), ("B", 2)]` hace que se explore primero `B`
y el resultado pasa a ser `A → B → E → H` con costo 5 en lugar de 6. El camino y el
costo dependen del orden de escritura del grafo.

En BFS el efecto es menor. El orden dentro de cada nivel cambia, y con él cuál de los
caminos mínimos se reporta, pero la longitud en aristas se mantiene: con el orden
invertido devuelve `A → C → E → H`, también de tres aristas. Ni UCS ni A* se ven
afectados, porque el desempate lo resuelve la prioridad y no la posición en la lista.

## 11. Casos que el grafo del enunciado no cubre

El grafo de la guía es acíclico, tiene solución y es lo bastante pequeño como para
que casi cualquier implementación parezca correcta. Estos tres casos ejercitan lo
que ahí no se ve:

- **Grafo con ciclos:** comprueba que el control de estados explorados evita que la
  búsqueda entre en un bucle infinito.
- **Grafo sin solución:** la meta existe pero no es alcanzable. Los algoritmos deben
  vaciar la frontier y devolver `None`, no fallar.
- **Trampa para GBF:** un grafo diseñado para que la heurística engañe. Desde `S`,
  `h(P)=1` es menor que `h(Q)=2`, pero salir de `P` cuesta 20 y salir de `Q` cuesta 2.
  Es la pregunta 3 llevada a un caso donde la diferencia es evidente.

In [ ]:
# Aristas en ambos sentidos: sin control de explorados la búsqueda no terminaría.
cyclic_graph = {
    "A": [("B", 1), ("C", 4)],
    "B": [("A", 1), ("C", 1), ("D", 7)],
    "C": [("A", 4), ("B", 1), ("D", 2)],
    "D": [("B", 7), ("C", 2)]
}
cyclic_heuristic = {"A": 3, "B": 2, "C": 2, "D": 0}

# Z figura como nodo del grafo, pero ninguna arista llega hasta él.
disconnected_graph = {
    "A": [("B", 1)],
    "B": [("A", 1)],
    "Z": []
}

# GBF cae en S->P por su h bajo; UCS y A* prefieren S->Q, que es más barato.
trap_graph = {
    "S": [("P", 1), ("Q", 4)],
    "P": [("G", 20)],
    "Q": [("G", 2)],
    "G": []
}
trap_heuristic = {"S": 2, "P": 1, "Q": 2, "G": 0}

In [ ]:
print("GRAFO CON CICLOS (A -> D)")
compare(cyclic_graph, "A", "D", cyclic_heuristic)

print("\nGRAFO SIN SOLUCIÓN (A -> Z)")
compare(disconnected_graph, "A", "Z")

print("\nTRAMPA PARA GBF (S -> G)")
compare(trap_graph, "S", "G", trap_heuristic)

# Comprobaciones sobre los casos límite.
assert depth_first_search(disconnected_graph, "A", "Z", verbose=False) is None
assert breadth_first_search(disconnected_graph, "A", "Z", verbose=False) is None
assert uniform_cost_search(disconnected_graph, "A", "Z", verbose=False) is None

trap_gbf = greedy_best_first_search(trap_graph, trap_heuristic, "S", "G", verbose=False)
trap_astar = a_star_search(trap_graph, trap_heuristic, "S", "G", verbose=False)

assert trap_gbf["cost"] == 21, "GBF debería caer en la ruta cara S->P->G."
assert trap_astar["cost"] == 6, "A* debería encontrar S->Q->G."

print("\nLos tres casos se comportan como se esperaba.")

![Grafo DFS](https://drive.google.com/uc?export=view&id=1C-Gocif6ltAwtFuRoROrR7D5rfs7jjcc)

